<a href="https://colab.research.google.com/github/ElofssonLab/kb8029-book/blob/main/notebooks/session05-discussion-1.ipynb" style="display:inline-block;padding:10px 18px;background-color:#F9AB00;color:#000000;font-weight:bold;text-decoration:none;border-radius:6px;font-family:sans-serif;font-size:14px;">&#9654;&nbsp; Open in Google Colab</a>

# Session 5: in-class discussion problem (1 of 3)

This is the block 1 question of the lecture quiz. First guess the answer in the quiz, on your own.
Then run the code below and look at what it does. There is one mistake in it, but finding that is
only the way in: the point is to work out why the result comes out as it does. Discuss that with
your neighbour, then answer the quiz again.

## How many neighbour words does GKV have?

BLAST seeds its search with short words. For each three-letter word of the query, it lists every
word that scores at least $T$ against it with BLOSUM62: the *neighbour words*. The book page found
49 neighbour words for the query word `LWG` at $T = 12$. The word `GKV` sits right next to it in
haemoglobin beta (`...TALWGKVNVDEV...`) and has the same length.

At the same threshold $T = 12$, does `GKV` have about as many neighbour words as `LWG`, more, or
far fewer?

**Fix and run.** The cell below counts the neighbour words of both. One character in the line marked
`FIX` is wrong. Run it once as it is: what does it count? Then fix the line and run it again.

In [1]:
# Install the libraries Colab does not come with (does nothing if they are already installed)
import importlib.util, subprocess, sys
for module, package in [("Bio", "biopython")]:
    if importlib.util.find_spec(module) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=True)

In [2]:
from itertools import product
from Bio.Align import substitution_matrices

blosum62 = substitution_matrices.load("BLOSUM62")
AA = sorted(set(blosum62.alphabet) - {"*", "X", "B", "Z"})     # the 20 amino acids

def word_score(w1, w2):
    """The BLOSUM62 score of word w1 aligned to word w2, position by position."""
    return sum(blosum62[a, a] for a, b in zip(w1, w2))       # FIX: one letter is wrong

T = 12
for query_word in ["LWG", "GKV"]:
    words = ["".join(w) for w in product(AA, repeat=3)]       # all 8000 three-letter words
    neighbours = [w for w in words if word_score(query_word, w) >= T]
    print(f"{query_word}: self-score {word_score(query_word, query_word)}, "
          f"{len(neighbours)} neighbour words at T = {T}")

LWG: self-score 21.0, 8000 neighbour words at T = 12
GKV: self-score 15.0, 8000 neighbour words at T = 12


**If you finish early.**

1. Print BLOSUM62's diagonal scores, `blosum62[r, r]`, for L, W, G, K and V. Which one stands out?
2. Lower the threshold for `GKV` until it has about as many neighbour words as `LWG`. Which $T$ is
   that, and what would it cost BLAST in speed?

<details>
<summary><b>Discussion point</b> &mdash; open this once you have run the code and talked it through</summary>

As it was, `word_score` added the query's own diagonal scores,
`blosum62[a, a]`, whatever the other word was. Every one of the 8000 words then scored the
query's self-score (21 for `LWG`, 15 for `GKV`), so all 8000 cleared $T = 12$ for both. The fix
compares the two words: `blosum62[a, b]`.

Then `LWG` has 49 neighbour words and `GKV` only 5. Tryptophan's diagonal score is +11, far above
every other residue (cysteine is next at +9; most are +4 to +6). W is bulky and rare, so an exact
match at a W is much more surprising, and therefore more informative, than a match at a common
residue. `LWG` starts from a self-score of 4 + 11 + 6 = 21, nine points above the threshold, so many
substituted words still reach 12. `GKV` starts from 6 + 5 + 4 = 15, only three points above it.

The number of seeds BLAST finds therefore depends on the query itself: words with rare,
high-scoring residues give many seeds, and words made of common residues give few. Lowering $T$
gives more seeds, and a slower search.

</details>